# f6_m00_ejecucion.ipynb

## Qué hace
Orquestador visual de la Fase 6. Lanza cada submodulo en secuencia usando
`nbconvert --execute` y actualiza una tabla de progreso con ipywidgets en tiempo real.

## Requisitos
- `src/utils/orquestador.py` con `orquestador_fase()` y `ejecutar_notebook()`
- `src/utils/progress.py` con `progreso_fase`
- Todos los notebooks de Fase 6 existentes en el mismo directorio
- `nbconvert` instalado: `pip install nbconvert`

## Genera
- Ejecuta y sobreescribe en-place cada notebook de Fase 6
- `docs/html/fase6/m00_ejecucion.html` (página HTML de este módulo)

## Flujo
Celda config → instanciar orquestador → bucle ejecutar notebooks → render HTML

## Siguiente
Revisar resultados de cada submodulo individualmente.

In [ ]:
# 1. CONFIGURACIÓN DE RUTAS
import sys
from pathlib import Path

def _encontrar_root(start: Path) -> Path:
    for parent in [start] + list(start.parents):
        if (parent / 'src').is_dir():
            return parent
    raise FileNotFoundError('No se encontró src/ subiendo desde ' + str(start))

ROOT = _encontrar_root(Path.cwd())
sys.path.insert(0, str(ROOT))

# Directorio donde viven los notebooks de Fase 6
# Ajustar si los notebooks están en una subcarpeta distinta
DIR_NOTEBOOKS = Path.cwd()

print(f'ROOT: {ROOT}')
print(f'DIR_NOTEBOOKS: {DIR_NOTEBOOKS}')

In [ ]:
# 2. IMPORTS
from src.utils.orquestador import orquestador_fase, ejecutar_notebook
from src.html.render import render_pagina

In [ ]:
# 3. LISTA DE NOTEBOOKS A EJECUTAR
# Editar esta lista para incluir/excluir módulos o cambiar el orden.
# Los notebooks de gestión (m00_prep) ya deben estar ejecutados antes.

NOTEBOOKS = [
    # --- SHAP ---
    ('m01a', 'f6_m01a_shap_global.ipynb'),
    ('m01b', 'f6_m01b_shap_local.ipynb'),
    ('m01c', 'f6_m01c_shap_cohortes.ipynb'),
    ('m01d', 'f6_m01d_shapash.ipynb'),
    # --- Interpretabilidad Alternativa ---
    ('m02a', 'f6_m02a_lime.ipynb'),
    ('m02b', 'f6_m02b_dice.ipynb'),
    # --- Fairness y Errores ---
    ('m03a', 'f6_m03a_fairness.ipynb'),
    ('m03b', 'f6_m03b_errores_fpfn.ipynb'),
    # --- Robustez y Calibración ---
    ('m04a', 'f6_m04a_stress.ipynb'),
    ('m04b', 'f6_m04b_calibracion.ipynb'),
    ('m04c', 'f6_m04c_sostenibilidad.ipynb'),
    # --- Informe Final ---
    ('m05',  'f6_m05_informe_final.ipynb'),
]

# Timeout por celda en segundos (SHAP sobre test completo puede tardar)
TIMEOUT_POR_CELDA = 3600   # 1 hora

print(f'{len(NOTEBOOKS)} notebooks configurados.')

In [ ]:
# 4. INSTANCIAR ORQUESTADOR Y MOSTRAR TABLA
orch = orquestador_fase("Fase 6 — Interpretabilidad y Evaluación Final")
orch.mostrar()

In [ ]:
# 5. EJECUTAR NOTEBOOKS EN SECUENCIA
# La tabla de progreso se actualiza en tiempo real.
# Si un notebook falla, se marca con ❌ y se continúa con el siguiente.

errores = []

for modulo_id, nombre_nb in NOTEBOOKS:
    ruta_nb = DIR_NOTEBOOKS / nombre_nb

    # Verificar que el fichero existe antes de intentar ejecutarlo
    if not ruta_nb.exists():
        print(f'⚠️  {nombre_nb} no encontrado — saltando.')
        orch.error(modulo_id)
        errores.append((modulo_id, nombre_nb, 'fichero no encontrado'))
        continue

    orch.iniciar(modulo_id)
    ok = ejecutar_notebook(ruta_nb, timeout=TIMEOUT_POR_CELDA)

    if ok:
        orch.ok(modulo_id)
    else:
        orch.error(modulo_id)
        errores.append((modulo_id, nombre_nb, 'error en ejecución'))

# Resumen final
n_ok = len(NOTEBOOKS) - len(errores)
print(f'\n✅ Completados: {n_ok}/{len(NOTEBOOKS)}')
if errores:
    print('\n❌ Módulos con error:')
    for mid, nb, motivo in errores:
        print(f'   {mid}: {nb} — {motivo}')

In [ ]:
# 6. GENERAR HTML DE ESTE MÓDULO
n_ok    = len(NOTEBOOKS) - len(errores)
n_total = len(NOTEBOOKS)
pct     = int(n_ok / n_total * 100) if n_total else 0

# Construir tabla resumen para el HTML
filas_resumen = ''
for mid, nb, *_ in [(m, n) for m, n in NOTEBOOKS]:
    fue_error = any(e[0] == mid for e in errores)
    icono = '❌' if fue_error else '✅'
    bg    = '#fff5f5' if fue_error else '#f0fff4'
    filas_resumen += f"""
    <tr style="background:{bg}">
        <td style="padding:8px 12px">{icono}</td>
        <td style="padding:8px 12px; font-family:monospace; font-size:12px">{mid}</td>
        <td style="padding:8px 12px; font-size:13px">{nb}</td>
    </tr>"""

barra_color = '#38a169' if pct == 100 else '#3182ce'

contenido_html = f"""
<h2 style="color:#2d3748">▶️ Orquestador de Fase 6</h2>
<p style="color:#4a5568; font-size:14px">
    Ejecuta todos los submodulos de Fase 6 en secuencia usando
    <code>nbconvert --execute</code> con barra de progreso visual.
</p>

<div style="margin:20px 0; padding:16px; background:#f7fafc; border-radius:8px">
    <div style="font-size:14px; color:#4a5568; margin-bottom:8px">
        Progreso: {n_ok}/{n_total} módulos ({pct}%)
    </div>
    <div style="background:#e2e8f0; border-radius:4px; height:10px; width:100%">
        <div style="background:{barra_color}; border-radius:4px; height:10px; width:{pct}%"></div>
    </div>
</div>

<table style="width:100%; border-collapse:collapse; font-size:13px">
    <thead>
        <tr style="background:#edf2f7">
            <th style="padding:8px 12px; text-align:left">Estado</th>
            <th style="padding:8px 12px; text-align:left">ID</th>
            <th style="padding:8px 12px; text-align:left">Notebook</th>
        </tr>
    </thead>
    <tbody>{filas_resumen}</tbody>
</table>
"""

ruta_html = render_pagina(
    nombre_fichero=__file__ if '__file__' in dir() else 'f6_m00_ejecucion.ipynb',
    contenido_html=contenido_html,
)
print(f'✅ HTML generado: {ruta_html}')